In [ ]:

# ============================================
# STEP 1: INITIALIZE EXPERIMENT ENVIRONMENT
# ============================================

from pathlib import Path
import pandas as pd
import boto3

from PIL import Image
from io import BytesIO

# --------------------------------------------
# 1. Find project root
# --------------------------------------------

relative_sample_path = Path(
    "reports/eda/enhancement_experiments/screening_121_images.csv"
)

project_root = None

for candidate in [Path.cwd(), *Path.cwd().parents]:

    if (candidate / relative_sample_path).exists():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError(
        "Cannot find screening_121_images.csv. "
        "Check that the previous notebook saved it "
        "inside reports/eda/enhancement_experiments."
    )

print("Project root:", project_root)

# --------------------------------------------
# 2. Define experiment directories
# --------------------------------------------

output_dir = (
    project_root /
    "reports/eda/enhancement_experiments"
)

cache_dir = output_dir / "original_cache"

cache_dir.mkdir(
    parents=True,
    exist_ok=True
)

# --------------------------------------------
# 3. Load the existing 121-image sample
# --------------------------------------------

sample_csv = output_dir / "screening_121_images.csv"

screening_sample = pd.read_csv(sample_csv)

# Validate columns
required_columns = [
    "file_name",
    "s3_key",
    "folder_1",
    "folder_2"
]

missing_columns = [
    col for col in required_columns
    if col not in screening_sample.columns
]

assert not missing_columns, (
    f"Missing columns: {missing_columns}"
)

assert len(screening_sample) == 121

# --------------------------------------------
# 4. Initialize AWS S3 connection
# --------------------------------------------

BUCKET_NAME = (
    "amal-stitchingnet-data-903473565738-ap-southeast-2-an"
)

s3 = boto3.client(
    "s3",
    region_name="ap-southeast-2"
)

# --------------------------------------------
# 5. Display experiment information
# --------------------------------------------

print("\nExperiment environment initialized")
print("-----------------------------------")

print("Total images:", len(screening_sample))

print(
    "Fabric categories:",
    screening_sample["folder_1"].nunique()
)

print(
    "Defect categories:",
    screening_sample["folder_2"].nunique()
)

print("Output directory:", output_dir)
print("Cache directory:", cache_dir)
print("S3 bucket:", BUCKET_NAME)

display(screening_sample.head())


In [ ]:

# ============================================
# STEP 2: DOWNLOAD AND CACHE ORIGINAL IMAGES
# ============================================

downloaded_count = 0
local_paths = []

for row in screening_sample.itertuples(index=False):

    local_path = cache_dir / row.file_name

    if not local_path.exists():

        s3.download_file(
            BUCKET_NAME,
            row.s3_key,
            str(local_path)
        )

        downloaded_count += 1

    # Verify image integrity
    with Image.open(local_path) as img:
        img.verify()

    local_paths.append(str(local_path))

# Add local paths to the experiment dataframe
screening_sample["local_path"] = local_paths

# Save experiment manifest
screening_sample.to_csv(
    output_dir / "screening_121_images_with_paths.csv",
    index=False
)

print("\nImage caching completed")
print("------------------------")
print("Selected images:", len(screening_sample))
print("New downloads:", downloaded_count)
print("Available images:", len(local_paths))
print("Cache directory:", cache_dir)


In [ ]:

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image

print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)


In [ ]:

# ============================================
# IMAGE ENHANCEMENT LABORATORY
# ============================================

def normalize_image(image):

    image = np.asarray(
        image,
        dtype=np.float32
    )

    return cv2.normalize(
        image,
        None,
        0,
        255,
        cv2.NORM_MINMAX
    ).astype(np.uint8)


def apply_clahe_lab(rgb):

    lab = cv2.cvtColor(
        rgb,
        cv2.COLOR_RGB2LAB
    )

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    lab[:, :, 0] = clahe.apply(
        lab[:, :, 0]
    )

    return cv2.cvtColor(
        lab,
        cv2.COLOR_LAB2RGB
    )


def apply_unsharp(rgb):

    blurred = cv2.GaussianBlur(
        rgb,
        (0, 0),
        sigmaX=1.0
    )

    sharpened = cv2.addWeighted(
        rgb,
        1.5,
        blurred,
        -0.5,
        0
    )

    return sharpened


def apply_scharr(gray):

    gx = cv2.Scharr(
        gray,
        cv2.CV_32F,
        1,
        0
    )

    gy = cv2.Scharr(
        gray,
        cv2.CV_32F,
        0,
        1
    )

    magnitude = cv2.magnitude(
        gx,
        gy
    )

    return normalize_image(magnitude)


def apply_canny(gray):

    median = np.median(gray)

    lower = int(max(0, 0.66 * median))
    upper = int(min(255, 1.33 * median))

    return cv2.Canny(
        gray,
        lower,
        upper
    )


def apply_gabor(gray):

    responses = []

    angles = [
        0,
        np.pi / 4,
        np.pi / 2,
        3 * np.pi / 4
    ]

    for angle in angles:

        kernel = cv2.getGaborKernel(
            (21, 21),
            sigma=4.0,
            theta=angle,
            lambd=8.0,
            gamma=0.5,
            psi=0,
            ktype=cv2.CV_32F
        )

        response = cv2.filter2D(
            gray,
            cv2.CV_32F,
            kernel
        )

        responses.append(
            np.abs(response)
        )

    combined = np.maximum.reduce(
        responses
    )

    return normalize_image(combined)


def apply_morphology(gray):

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (5, 5)
    )

    top_hat = cv2.morphologyEx(
        gray,
        cv2.MORPH_TOPHAT,
        kernel
    )

    black_hat = cv2.morphologyEx(
        gray,
        cv2.MORPH_BLACKHAT,
        kernel
    )

    return top_hat, black_hat


def apply_clahe_scharr(rgb):

    lab = cv2.cvtColor(
        rgb,
        cv2.COLOR_RGB2LAB
    )

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    enhanced_l = clahe.apply(
        lab[:, :, 0]
    )

    return apply_scharr(enhanced_l)


def apply_bilateral_clahe(rgb):

    denoised = cv2.bilateralFilter(
        rgb,
        d=5,
        sigmaColor=30,
        sigmaSpace=30
    )

    return apply_clahe_lab(denoised)


# ============================================
# MAIN TRANSFORMATION FUNCTION
# ============================================

def create_representations(rgb):

    gray = cv2.cvtColor(
        rgb,
        cv2.COLOR_RGB2GRAY
    )

    hsv = cv2.cvtColor(
        rgb,
        cv2.COLOR_RGB2HSV
    )

    top_hat, black_hat = apply_morphology(
        gray
    )

    representations = {

        "01_original": rgb,

        "02_lab_clahe": apply_clahe_lab(
            rgb
        ),

        "03_hsv_hue": hsv[:, :, 0],

        "04_hsv_saturation": hsv[:, :, 1],

        "05_hsv_value": hsv[:, :, 2],

        "06_unsharp": apply_unsharp(
            rgb
        ),

        "07_scharr": apply_scharr(
            gray
        ),

        "08_canny": apply_canny(
            gray
        ),

        "09_gabor": apply_gabor(
            gray
        ),

        "10_top_hat": top_hat,

        "11_black_hat": black_hat,

        "12_clahe_scharr": apply_clahe_scharr(
            rgb
        ),

        "13_bilateral_clahe": apply_bilateral_clahe(
            rgb
        )
    }

    return representations


print("Enhancement laboratory created.")
print("Number of representations: 13")


In [ ]:

# Select representative defect categories

pilot_classes = [
    "6. Puckering",
    "7. Stain and damage",
    "8. Needle mark"
]

pilot_sample = screening_sample[
    screening_sample["folder_2"].isin(
        pilot_classes
    )
].copy()

assert len(pilot_sample) == 33

# Select one image per defect category and fabric
pilot_sample = (
    pilot_sample
    .sort_values(
        ["folder_2", "folder_1"]
    )
)

# One sample per category for initial visualization
pilot_examples = (
    pilot_sample
    .groupby("folder_2", group_keys=False)
    .head(1)
)

display(
    pilot_examples[
        ["folder_1", "folder_2", "file_name"]
    ]
)


In [ ]:

# ============================================
# PILOT VISUALIZATION
# ============================================

for row in pilot_examples.itertuples(index=False):

    rgb = np.array(
        Image.open(
            row.local_path
        ).convert("RGB")
    )

    representations = create_representations(
        rgb
    )

    fig, axes = plt.subplots(
        4,
        4,
        figsize=(16, 15)
    )

    fig.suptitle(
        f"{row.folder_2} | "
        f"{row.folder_1} | "
        f"{row.file_name}",
        fontsize=15
    )

    for ax, (name, result) in zip(
        axes.flat,
        representations.items()
    ):

        if result.ndim == 2:

            ax.imshow(
                result,
                cmap="gray",
                vmin=0,
                vmax=255
            )

        else:

            ax.imshow(result)

        ax.set_title(
            name,
            fontsize=9
        )

        ax.axis("off")

    # Hide unused subplot
    for ax in axes.flat[len(representations):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:

# ============================================
# STEP 5: PROCESS ALL 121 IMAGES
# ============================================

from PIL import Image
from pathlib import Path
import numpy as np

representation_dir = output_dir / "representations"

representation_dir.mkdir(
    parents=True,
    exist_ok=True
)

assert len(screening_sample) == 121

assert screening_sample["local_path"].apply(
    lambda p: Path(p).exists()
).all()

processed_count = 0
method_names = None

for row in screening_sample.itertuples(index=False):

    # Load untouched original
    rgb = np.array(
        Image.open(
            row.local_path
        ).convert("RGB")
    )

    # Generate all representations once
    representations = create_representations(rgb)

    if method_names is None:
        method_names = list(representations.keys())

    # Save each representation
    for method_name, result in representations.items():

        method_dir = (
            representation_dir / method_name
        )

        method_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        save_path = (
            method_dir /
            f"{Path(row.file_name).stem}.png"
        )

        Image.fromarray(
            np.asarray(result, dtype=np.uint8)
        ).save(save_path)

    processed_count += 1

    if processed_count % 20 == 0:

        print(
            f"Processed {processed_count}/121 images"
        )

print("\nFULL EXPERIMENT COMPLETED")
print("-------------------------")
print("Original images:", processed_count)
print("Methods:", len(method_names))
print(
    "Total representations:",
    processed_count * len(method_names)
)
print("Output:", representation_dir)


In [ ]:

# ============================================
# STEP 6: GENERATE 13 CONTACT SHEETS
# ============================================

import matplotlib.pyplot as plt
import cv2
import numpy as np

from PIL import Image
from pathlib import Path

contact_dir = output_dir / "contact_sheets"

contact_dir.mkdir(
    parents=True,
    exist_ok=True
)

# Sort classes numerically
class_order = sorted(
    screening_sample["folder_2"].unique(),
    key=lambda x: int(x.split(".")[0])
)

# Sort fabrics by their letter codes
fabric_order = sorted(
    screening_sample["folder_1"].unique()
)

# Lookup every image using its class/fabric combination
image_lookup = {
    (row.folder_2, row.folder_1): row
    for row in screening_sample.itertuples(index=False)
}

assert len(image_lookup) == 121


def prepare_for_display(
    method_name,
    result,
    original_path
):

    result = np.asarray(result)

    # Morphological outputs can have very small values.
    # Normalize for visualization only.
    if method_name in [
        "10_top_hat",
        "11_black_hat"
    ]:

        result = cv2.normalize(
            result.astype(np.float32),
            None,
            0,
            255,
            cv2.NORM_MINMAX
        ).astype(np.uint8)

    # Hue is unreliable where saturation is very low.
    # Mask those pixels in the visualization.
    if method_name == "03_hsv_hue":

        original = np.array(
            Image.open(original_path).convert("RGB")
        )

        hsv = cv2.cvtColor(
            original,
            cv2.COLOR_RGB2HSV
        )

        result = result.copy()
        result[hsv[:, :, 1] < 25] = 0

    return result


# Generate one contact sheet per method
for method_name in method_names:

    fig, axes = plt.subplots(
        nrows=11,
        ncols=11,
        figsize=(22, 22)
    )

    for row_idx, class_name in enumerate(class_order):

        for col_idx, fabric in enumerate(fabric_order):

            ax = axes[row_idx, col_idx]

            record = image_lookup[
                (class_name, fabric)
            ]

            image_path = (
                representation_dir /
                method_name /
                f"{Path(record.file_name).stem}.png"
            )

            result = np.array(
                Image.open(image_path)
            )

            result = prepare_for_display(
                method_name,
                result,
                record.local_path
            )

            if result.ndim == 2:

                ax.imshow(
                    result,
                    cmap="gray",
                    vmin=0,
                    vmax=179 if method_name == "03_hsv_hue" else 255,
                    interpolation="nearest"
                )

            else:

                ax.imshow(
                    result,
                    interpolation="nearest"
                )

            # Filename above every image
            ax.set_title(
                record.file_name,
                fontsize=4.5,
                pad=2
            )

            # Fabric identification on first row
            if row_idx == 0:

                fabric_code = fabric.split(".")[0]

                ax.text(
                    0.5,
                    1.35,
                    fabric_code,
                    transform=ax.transAxes,
                    ha="center",
                    fontsize=8,
                    fontweight="bold"
                )

            # Class identification on first column
            if col_idx == 0:

                ax.text(
                    -0.12,
                    0.5,
                    class_name,
                    transform=ax.transAxes,
                    ha="right",
                    va="center",
                    fontsize=6
                )

            ax.set_xticks([])
            ax.set_yticks([])

    fig.suptitle(
        f"StitchingNet Enhancement Experiment: {method_name}",
        fontsize=15
    )

    fig.subplots_adjust(
        left=0.13,
        right=0.99,
        top=0.96,
        bottom=0.02,
        wspace=0.04,
        hspace=0.18
    )

    save_path = (
        contact_dir /
        f"{method_name}.png"
    )

    fig.savefig(
        save_path,
        dpi=140,
        bbox_inches="tight"
    )

    plt.close(fig)

    print("Saved:", save_path)

print("\nAll 13 contact sheets generated.")


In [ ]:

from IPython.display import display, Image as NotebookImage

def show_contact_sheet(method_name):

    image_path = contact_dir / f"{method_name}.png"

    if not image_path.exists():
        print("Contact sheet not found:", method_name)
        return

    print("Viewing:", method_name)

    display(
        NotebookImage(
            filename=str(image_path)
        )
    )


# First comparison group
show_contact_sheet("01_original")
show_contact_sheet("02_lab_clahe")
show_contact_sheet("06_unsharp")
show_contact_sheet("07_scharr")
show_contact_sheet("08_canny")
show_contact_sheet("09_gabor")
show_contact_sheet("10_top_hat")
show_contact_sheet("11_black_hat")
show_contact_sheet("12_clahe_scharr")
show_contact_sheet("13_bilateral_clahe")


In [ ]:
show_contact_sheet("03_hsv_hue")
show_contact_sheet("04_hsv_saturation")
show_contact_sheet("05_hsv_value")

In [ ]:
# ==========================================
# RECOVER DATASET INVESTIGATION VARIABLES
# ==========================================

import boto3
import pandas as pd
from pathlib import PurePosixPath

# AWS configuration
BUCKET_NAME = "amal-stitchingnet-data-903473565738-ap-southeast-2-an"
REGION = "ap-southeast-2"
DATA_PREFIX = "data/"

# Connect to S3 using SageMaker execution role
s3 = boto3.client(
    "s3",
    region_name=REGION
)

print("Connected to S3")

# List all dataset objects
paginator = s3.get_paginator("list_objects_v2")

all_objects = []

for page in paginator.paginate(
    Bucket=BUCKET_NAME,
    Prefix=DATA_PREFIX
):
    all_objects.extend(
        page.get("Contents", [])
    )

print("Total objects:", len(all_objects))

# Rebuild inventory
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png",
    ".bmp", ".tif", ".tiff", ".webp"
}

records = []

for obj in all_objects:

    key = obj["Key"]

    if key.endswith("/"):
        continue

    path = PurePosixPath(key)

    folders = path.parts[1:-1]

    records.append({
        "file_name": path.name,
        "s3_key": key,
        "extension": path.suffix.lower(),
        "file_size_kb": obj["Size"] / 1024,
        "last_modified": obj["LastModified"],

        "folder_1": (
            folders[0] if len(folders) > 0 else None
        ),

        "folder_2": (
            folders[1] if len(folders) > 1 else None
        ),

        "is_image": (
            path.suffix.lower() in IMAGE_EXTENSIONS
        )
    })

inventory_df = pd.DataFrame(records)

# Recreate image dataframe
image_df = inventory_df[
    inventory_df["is_image"]
].copy()

# Recreate class distribution
class_distribution = (
    image_df.groupby("folder_2")
    .size()
    .reset_index(name="image_count")
)

class_distribution["class_id"] = (
    class_distribution["folder_2"]
    .str.extract(r"^(\d+)")
    .astype(int)
)

class_distribution = (
    class_distribution
    .sort_values("class_id")
    .reset_index(drop=True)
)

# Recreate fabric and class order
fabric_order = sorted(
    image_df["folder_1"].unique()
)

class_order = (
    class_distribution["folder_2"]
    .tolist()
)

# Select three images from each combination
combination_counts = (
    image_df.groupby(["folder_1", "folder_2"])
    .size()
)

assert combination_counts.min() >= 3, (
    "Some combinations have fewer than 3 images"
)

stratified_sample = (
    image_df
    .groupby(
        ["folder_1", "folder_2"],
        group_keys=False
    )
    .sample(
        n=3,
        random_state=42
    )
    .reset_index(drop=True)
)

# Verify everything
print("\nRECOVERY SUCCESSFUL")
print("-------------------")

print("Total images:", len(image_df))
print("Fabric categories:", len(fabric_order))
print("Stitching classes:", len(class_order))
print("Fabric/class combinations:", len(combination_counts))
print("Selected images:", len(stratified_sample))

display(
    stratified_sample[
        ["folder_1", "folder_2", "file_name"]
    ].head(15)
)

In [ ]:

from pathlib import Path
from io import BytesIO
import hashlib
import re

import cv2
import numpy as np
import pandas as pd

# ============================================
# EXPERIMENT 14 - STEP 2
# MULTI-VIEW PREPROCESSING
# ============================================

# Use the original full dataset.
# Do not use screening_sample here.

assert "image_df" in globals(), (
    "image_df is missing. Run the original "
    "dataset loading cells."
)

assert "s3" in globals(), (
    "S3 client is missing."
)

assert "BUCKET_NAME" in globals(), (
    "BUCKET_NAME is missing."
)

# Output directory
BASE_DIR = Path(
    "data/processed/exp14_multiview"
)

ORIGINAL_DIR = BASE_DIR / "original"
LAB_DIR = BASE_DIR / "lab_clahe"
BILATERAL_DIR = BASE_DIR / "bilateral_clahe"

for directory in [
    ORIGINAL_DIR,
    LAB_DIR,
    BILATERAL_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

# ============================================
# Enhancement parameters
# ============================================

CLAHE_CLIP_LIMIT = 2.0
CLAHE_TILE_GRID = (8, 8)

BILATERAL_DIAMETER = 5
BILATERAL_SIGMA_COLOR = 30
BILATERAL_SIGMA_SPACE = 30

# First run: process only 20 images.
# After verification, change to None
# to process the complete dataset.

MAX_IMAGES = 20

print("Preprocessing configuration ready.")
print("Output directory:", BASE_DIR)
print("CLAHE:", CLAHE_CLIP_LIMIT, CLAHE_TILE_GRID)
print(
    "Bilateral:",
    BILATERAL_DIAMETER,
    BILATERAL_SIGMA_COLOR,
    BILATERAL_SIGMA_SPACE
)


In [ ]:

# ============================================
# LAB CLAHE
# ============================================

clahe = cv2.createCLAHE(
    clipLimit=CLAHE_CLIP_LIMIT,
    tileGridSize=CLAHE_TILE_GRID
)


def apply_lab_clahe(bgr_image):

    lab = cv2.cvtColor(
        bgr_image,
        cv2.COLOR_BGR2LAB
    )

    l_channel, a_channel, b_channel = (
        cv2.split(lab)
    )

    enhanced_l = clahe.apply(
        l_channel
    )

    enhanced_lab = cv2.merge([
        enhanced_l,
        a_channel,
        b_channel
    ])

    enhanced_bgr = cv2.cvtColor(
        enhanced_lab,
        cv2.COLOR_LAB2BGR
    )

    return enhanced_bgr


# ============================================
# BILATERAL + LAB CLAHE
# ============================================

def apply_bilateral_clahe(bgr_image):

    lab = cv2.cvtColor(
        bgr_image,
        cv2.COLOR_BGR2LAB
    )

    l_channel, a_channel, b_channel = (
        cv2.split(lab)
    )

    # Reduce small-scale noise while
    # attempting to preserve important edges.

    filtered_l = cv2.bilateralFilter(
        l_channel,
        d=BILATERAL_DIAMETER,
        sigmaColor=BILATERAL_SIGMA_COLOR,
        sigmaSpace=BILATERAL_SIGMA_SPACE
    )

    # Improve local contrast after filtering.

    enhanced_l = clahe.apply(
        filtered_l
    )

    enhanced_lab = cv2.merge([
        enhanced_l,
        a_channel,
        b_channel
    ])

    enhanced_bgr = cv2.cvtColor(
        enhanced_lab,
        cv2.COLOR_LAB2BGR
    )

    return enhanced_bgr


print("Both enhancement functions are ready.")


In [ ]:

def apply_bilateral_clahe(bgr_image):

    # Apply bilateral filtering to the complete
    # colour image, as in our 121-image experiment.

    filtered = cv2.bilateralFilter(
        bgr_image,
        d=BILATERAL_DIAMETER,
        sigmaColor=BILATERAL_SIGMA_COLOR,
        sigmaSpace=BILATERAL_SIGMA_SPACE
    )

    # Then apply LAB CLAHE
    enhanced = apply_lab_clahe(filtered)

    return enhanced


print("Bilateral + CLAHE function updated.")


In [ ]:

# ============================================
# EXPERIMENT 14 - STEP 2
# PROCESS FIRST 20 IMAGES
# ============================================

# Create full source dataset
source_df = (
    image_df[
        [
            "file_name",
            "s3_key",
            "folder_1",
            "folder_2"
        ]
    ]
    .dropna()
    .drop_duplicates(subset=["s3_key"])
    .copy()
)

# Select 20 reproducible random images
if MAX_IMAGES is not None:

    source_df = source_df.sample(
        n=min(MAX_IMAGES, len(source_df)),
        random_state=42
    )

source_df = source_df.reset_index(drop=True)

print("Images selected:", len(source_df))


# --------------------------------------------
# Helper for safe folder names
# --------------------------------------------

def safe_name(value):

    return re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        str(value)
    ).strip("_")


# --------------------------------------------
# Processing
# --------------------------------------------

records = []
errors = []

for idx, row in source_df.iterrows():

    fabric = safe_name(row["folder_1"])
    defect = safe_name(row["folder_2"])

    filename = Path(row["file_name"]).name
    stem = Path(filename).stem
    extension = Path(filename).suffix.lower()

    # Unique identifier from S3 key
    unique_id = hashlib.md5(
        row["s3_key"].encode()
    ).hexdigest()[:8]

    relative_dir = Path(fabric) / defect

    original_path = (
        ORIGINAL_DIR /
        relative_dir /
        f"{stem}_{unique_id}{extension}"
    )

    lab_path = (
        LAB_DIR /
        relative_dir /
        f"{stem}_{unique_id}.png"
    )

    bilateral_path = (
        BILATERAL_DIR /
        relative_dir /
        f"{stem}_{unique_id}.png"
    )

    for path in [
        original_path,
        lab_path,
        bilateral_path
    ]:
        path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

    try:

        # ------------------------------------
        # Load original image
        # ------------------------------------

        bgr = None

        if original_path.exists():

            bgr = cv2.imread(
                str(original_path),
                cv2.IMREAD_COLOR
            )

        # Download if original is missing
        # or cannot be decoded.

        if bgr is None:

            response = s3.get_object(
                Bucket=BUCKET_NAME,
                Key=row["s3_key"]
            )

            image_bytes = response["Body"].read()

            bgr = cv2.imdecode(
                np.frombuffer(
                    image_bytes,
                    dtype=np.uint8
                ),
                cv2.IMREAD_COLOR
            )

            if bgr is None:
                raise ValueError(
                    "Image decoding failed."
                )

            # Preserve exact original bytes
            original_path.write_bytes(
                image_bytes
            )

        # ------------------------------------
        # Generate LAB CLAHE
        # ------------------------------------

        if not lab_path.exists():

            lab_image = apply_lab_clahe(bgr)

            success, encoded = cv2.imencode(
                ".png",
                lab_image
            )

            if not success:
                raise ValueError(
                    "LAB encoding failed."
                )

            lab_path.write_bytes(
                encoded.tobytes()
            )

        # ------------------------------------
        # Generate Bilateral + CLAHE
        # ------------------------------------

        if not bilateral_path.exists():

            bilateral_image = (
                apply_bilateral_clahe(bgr)
            )

            success, encoded = cv2.imencode(
                ".png",
                bilateral_image
            )

            if not success:
                raise ValueError(
                    "Bilateral encoding failed."
                )

            bilateral_path.write_bytes(
                encoded.tobytes()
            )

        # ------------------------------------
        # Verify dimensions
        # ------------------------------------

        height, width = bgr.shape[:2]

        records.append({

            "file_name": filename,
            "s3_key": row["s3_key"],

            "folder_1": row["folder_1"],
            "folder_2": row["folder_2"],

            "width": width,
            "height": height,

            "original_path": str(original_path),
            "lab_clahe_path": str(lab_path),
            "bilateral_clahe_path": str(
                bilateral_path
            )

        })

    except Exception as e:

        errors.append({

            "file_name": filename,
            "s3_key": row["s3_key"],
            "error": str(e)

        })

    if (idx + 1) % 5 == 0:

        print(
            f"Progress: {idx + 1}/{len(source_df)} "
            f"| Success: {len(records)} "
            f"| Failed: {len(errors)}"
        )


# --------------------------------------------
# Save experiment records
# --------------------------------------------

processed_df = pd.DataFrame(records)

processed_df.to_csv(
    BASE_DIR / "processed_manifest.csv",
    index=False
)

if errors:

    pd.DataFrame(errors).to_csv(
        BASE_DIR / "processing_errors.csv",
        index=False
    )


print("\nPREPROCESSING FINISHED")
print("-----------------------")

print("Selected:", len(source_df))
print("Successful:", len(records))
print("Failed:", len(errors))

print("Manifest:")
print(BASE_DIR / "processed_manifest.csv")


In [ ]:

# ============================================
# VERIFY PROCESSED IMAGES
# ============================================

import matplotlib.pyplot as plt

assert len(processed_df) > 0, (
    "No images were processed successfully."
)

sample = processed_df.iloc[0]

original = cv2.imread(
    sample["original_path"]
)

lab = cv2.imread(
    sample["lab_clahe_path"]
)

bilateral = cv2.imread(
    sample["bilateral_clahe_path"]
)

assert original is not None
assert lab is not None
assert bilateral is not None

assert original.shape == lab.shape == bilateral.shape

fig, axes = plt.subplots(
    1, 3,
    figsize=(16, 6)
)

images = [
    ("Original RGB", original),
    ("LAB CLAHE", lab),
    ("Bilateral + CLAHE", bilateral)
]

for ax, (title, image) in zip(axes, images):

    ax.imshow(
        cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )
    )

    ax.set_title(title)
    ax.axis("off")

plt.suptitle(
    f"{sample['file_name']} | {sample['folder_2']}",
    fontsize=13
)

plt.tight_layout()
plt.show()

print("Original dimensions:", original.shape)
print("LAB dimensions:", lab.shape)
print("Bilateral dimensions:", bilateral.shape)
print("Visual verification completed.")


In [ ]:

# ============================================
# EXPERIMENT 14 - STEP 2
# PROCESSING QUALITY AUDIT
# ============================================

import shutil
import cv2
import pandas as pd
from pathlib import Path

assert len(processed_df) > 0, (
    "No successfully processed images found."
)

audit_results = []

for _, row in processed_df.iterrows():

    paths = {
        "original": Path(row["original_path"]),
        "lab_clahe": Path(row["lab_clahe_path"]),
        "bilateral_clahe": Path(
            row["bilateral_clahe_path"]
        )
    }

    images = {}
    all_exist = True

    for name, path in paths.items():

        if not path.exists():
            all_exist = False
            break

        images[name] = cv2.imread(
            str(path),
            cv2.IMREAD_COLOR
        )

    valid = (
        all_exist
        and all(image is not None
                for image in images.values())
    )

    same_dimensions = False

    if valid:

        shapes = [
            image.shape
            for image in images.values()
        ]

        same_dimensions = (
            len(set(shapes)) == 1
        )

    audit_results.append({

        "file_name": row["file_name"],
        "folder_1": row["folder_1"],
        "folder_2": row["folder_2"],

        "all_files_exist": all_exist,
        "all_images_readable": valid,
        "same_dimensions": same_dimensions,

        "passed": (
            valid and same_dimensions
        )

    })


audit_df = pd.DataFrame(audit_results)

print("PROCESSING AUDIT")
print("----------------")

print("Processed:", len(processed_df))
print("Passed:", audit_df["passed"].sum())
print("Failed:", (~audit_df["passed"]).sum())

print("\nProcessing errors:", len(errors))

display(audit_df.head(20))


# ============================================
# STORAGE ESTIMATION
# ============================================

branch_columns = [
    "original_path",
    "lab_clahe_path",
    "bilateral_clahe_path"
]

branch_sizes = {}

for column in branch_columns:

    total_size = sum(
        Path(path).stat().st_size
        for path in processed_df[column]
    )

    branch_sizes[column] = total_size


print("\nCURRENT CACHE SIZE")
print("------------------")

for name, size in branch_sizes.items():

    print(
        name,
        round(size / (1024 ** 2), 2),
        "MB"
    )

current_total = sum(branch_sizes.values())

average_per_image = (
    current_total / len(processed_df)
)

full_image_count = image_df["s3_key"].nunique()

estimated_total = (
    average_per_image * full_image_count
)

estimated_with_headroom = (
    estimated_total * 1.3
)

disk = shutil.disk_usage(BASE_DIR)

print("\nFULL DATASET ESTIMATE")
print("---------------------")

print("Total source images:", full_image_count)

print(
    "Estimated cache:",
    round(estimated_total / (1024 ** 3), 2),
    "GB"
)

print(
    "With 30% headroom:",
    round(estimated_with_headroom / (1024 ** 3), 2),
    "GB"
)

print(
    "Available disk:",
    round(disk.free / (1024 ** 3), 2),
    "GB"
)

if audit_df["passed"].all():

    print("\nIMAGE QUALITY AUDIT: PASSED")

else:

    print("\nIMAGE QUALITY AUDIT: FAILED")

if estimated_with_headroom < disk.free:

    print("STORAGE CHECK: SUFFICIENT")

else:

    print("STORAGE CHECK: INSUFFICIENT")


In [ ]:

# ============================================
# EXPERIMENT 15
# STEP 1 - LEAKAGE-SAFE DATASET SPLITTING
# ============================================

from pathlib import Path
import hashlib
import pandas as pd

from sklearn.model_selection import train_test_split

# ============================================
# 1. Validate source dataset
# ============================================

assert "image_df" in globals()

split_source = image_df.copy()

required = [
    "file_name",
    "s3_key",
    "folder_1",
    "folder_2"
]

assert all(
    col in split_source.columns
    for col in required
)

split_source = split_source.dropna(
    subset=required
).copy()

assert split_source["s3_key"].is_unique, (
    "Duplicate S3 keys detected."
)

# ============================================
# 2. Generate stable image IDs
# ============================================

split_source["sample_id"] = (
    split_source["s3_key"]
    .apply(
        lambda key: hashlib.sha256(
            key.encode("utf-8")
        ).hexdigest()[:16]
    )
)

# ============================================
# 3. Create stratification groups
# ============================================

split_source["stratum"] = (
    split_source["folder_1"].astype(str)
    + "||"
    + split_source["folder_2"].astype(str)
)

stratum_counts = (
    split_source["stratum"]
    .value_counts()
)

assert stratum_counts.min() >= 3, (
    "A fabric/class combination has fewer "
    "than 3 images."
)

print("Original images:", len(split_source))
print("Fabric categories:",
      split_source["folder_1"].nunique())
print("Stitching classes:",
      split_source["folder_2"].nunique())
print("Fabric/class combinations:",
      len(stratum_counts))

# ============================================
# 4. First split: 85% train+validation,
#    15% test
# ============================================

train_val_df, test_df = train_test_split(

    split_source,

    test_size=0.15,

    random_state=42,

    stratify=split_source["stratum"]

)

# ============================================
# 5. Second split: 70% train,
#    15% validation overall
# ============================================

train_df, val_df = train_test_split(

    train_val_df,

    test_size=(0.15 / 0.85),

    random_state=42,

    stratify=train_val_df["stratum"]

)

# ============================================
# 6. Assign split labels
# ============================================

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "validation"
test_df["split"] = "test"

split_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

# ============================================
# 7. Leakage and completeness checks
# ============================================

assert len(split_df) == len(split_source)

assert split_df["sample_id"].is_unique

train_ids = set(train_df["sample_id"])
val_ids = set(val_df["sample_id"])
test_ids = set(test_df["sample_id"])

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

# ============================================
# 8. Save manifest
# ============================================

SPLIT_DIR = Path(
    "reports/eda/exp15_dataset_splits"
)

SPLIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

manifest_path = (
    SPLIT_DIR / "dataset_split_manifest.csv"
)

split_df.to_csv(
    manifest_path,
    index=False
)

# ============================================
# 9. Summary
# ============================================

print("\nDATASET SPLITTING COMPLETED")
print("===========================")

print("Total:", len(split_df))

print("\nSplit distribution:")

display(
    split_df["split"]
    .value_counts()
    .to_frame("image_count")
)

print("\nClass distribution:")

display(
    pd.crosstab(
        split_df["folder_2"],
        split_df["split"]
    )
)

print("\nFabric distribution:")

display(
    pd.crosstab(
        split_df["folder_1"],
        split_df["split"]
    )
)

print("\nSaved:", manifest_path)

print("\nAll leakage and completeness checks passed.")


In [ ]:

# ============================================
# EXPERIMENT 16
# STEP 1 - SHA256 EXACT DUPLICATE DETECTION
# ============================================

import hashlib
import pandas as pd

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

assert "split_df" in globals()
assert "s3" in globals()

# ============================================
# Configuration
# ============================================

MAX_WORKERS = 8

OUTPUT_DIR = Path(
    "reports/eda/exp16_duplicate_detection"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================
# SHA256 calculation directly from S3
# ============================================

def calculate_s3_sha256(row):

    response = s3.get_object(
        Bucket=BUCKET_NAME,
        Key=row["s3_key"]
    )

    body = response["Body"]

    sha256 = hashlib.sha256()

    try:

        for chunk in body.iter_chunks(
            chunk_size=1024 * 1024
        ):

            if chunk:
                sha256.update(chunk)

    finally:
        body.close()

    return {
        "sample_id": row["sample_id"],
        "s3_key": row["s3_key"],
        "file_name": row["file_name"],
        "folder_1": row["folder_1"],
        "folder_2": row["folder_2"],
        "split": row["split"],
        "sha256": sha256.hexdigest()
    }


# ============================================
# Process all images
# ============================================

records = split_df[
    [
        "sample_id",
        "s3_key",
        "file_name",
        "folder_1",
        "folder_2",
        "split"
    ]
].to_dict("records")

hash_results = []
hash_errors = []

print("Starting SHA256 scan...")
print("Total images:", len(records))

with ThreadPoolExecutor(
    max_workers=MAX_WORKERS
) as executor:

    futures = {
        executor.submit(
            calculate_s3_sha256,
            row
        ): row
        for row in records
    }

    for index, future in enumerate(
        as_completed(futures),
        start=1
    ):

        try:

            hash_results.append(
                future.result()
            )

        except Exception as error:

            row = futures[future]

            hash_errors.append({
                "s3_key": row["s3_key"],
                "error": str(error)
            })

        if index % 500 == 0:

            print(
                f"Processed {index}/{len(records)}"
            )


# ============================================
# Save results
# ============================================

hash_df = pd.DataFrame(
    hash_results
)

hash_df.to_csv(
    OUTPUT_DIR / "sha256_manifest.csv",
    index=False
)

if hash_errors:

    pd.DataFrame(hash_errors).to_csv(
        OUTPUT_DIR / "hash_errors.csv",
        index=False
    )

    print("Hash errors:", len(hash_errors))

assert len(hash_df) == len(split_df), (
    "SHA256 scan incomplete. Resolve errors "
    "before continuing."
)

# ============================================
# Find exact duplicates
# ============================================

duplicate_df = hash_df[
    hash_df.duplicated(
        subset=["sha256"],
        keep=False
    )
].copy()

duplicate_groups = (
    duplicate_df
    .groupby("sha256")
    .agg(
        copies=("s3_key", "count"),
        different_classes=("folder_2", "nunique"),
        different_fabrics=("folder_1", "nunique"),
        different_splits=("split", "nunique"),
        files=(
            "file_name",
            lambda x: " | ".join(x.astype(str))
        )
    )
    .reset_index()
)

# ============================================
# Save duplicate report
# ============================================

duplicate_df.to_csv(
    OUTPUT_DIR / "exact_duplicate_images.csv",
    index=False
)

duplicate_groups.to_csv(
    OUTPUT_DIR / "duplicate_groups.csv",
    index=False
)

# ============================================
# Final report
# ============================================

print("\nSHA256 INVESTIGATION")
print("====================")

print("Total images:", len(hash_df))

print(
    "Unique image fingerprints:",
    hash_df["sha256"].nunique()
)

print(
    "Duplicate groups:",
    len(duplicate_groups)
)

print(
    "Images involved in duplicates:",
    len(duplicate_df)
)

print(
    "Extra duplicate copies:",
    len(duplicate_df)
    - len(duplicate_groups)
)

print(
    "Cross-class duplicate groups:",
    (
        duplicate_groups["different_classes"] > 1
    ).sum()
)

print(
    "Cross-split duplicate groups:",
    (
        duplicate_groups["different_splits"] > 1
    ).sum()
)

print("\nSaved reports:", OUTPUT_DIR)

if len(duplicate_groups) > 0:

    display(
        duplicate_groups.head(20)
    )

else:

    print("No exact duplicates found.")

assert len(hash_errors) == 0


In [ ]:

# ============================================
# EXPERIMENT 16
# STEP 2 - CORRECT CROSS-SPLIT DUPLICATES
# ============================================

# Join SHA256 fingerprints to original manifest

split_with_hash = split_df.merge(
    hash_df[["s3_key", "sha256"]],
    on="s3_key",
    how="left",
    validate="one_to_one"
)

assert split_with_hash["sha256"].notna().all()

# Choose the majority split for each hash.
# In a tie, prioritize test, then validation,
# then train.

def choose_split(series):

    counts = series.value_counts()

    maximum = counts.max()

    tied = set(
        counts[counts == maximum].index
    )

    for candidate in [
        "test",
        "validation",
        "train"
    ]:

        if candidate in tied:
            return candidate


target_split = (
    split_with_hash
    .groupby("sha256")["split"]
    .apply(choose_split)
)

split_with_hash["split"] = (
    split_with_hash["sha256"]
    .map(target_split)
)

# Verify no identical image crosses splits

assert (
    split_with_hash
    .groupby("sha256")["split"]
    .nunique()
    .max()
    == 1
)

# Replace active split manifest

split_df = (
    split_with_hash
    .drop(columns=["sha256"])
    .copy()
)

# Save corrected manifest

CORRECTED_PATH = (
    "reports/eda/exp16_duplicate_detection/"
    "dataset_split_manifest_corrected.csv"
)

split_df.to_csv(
    CORRECTED_PATH,
    index=False
)

print("DUPLICATE SPLIT CORRECTION COMPLETED")
print("====================================")

display(
    split_df["split"].value_counts()
)

print(
    "Remaining cross-split duplicate groups:",
    split_with_hash
    .groupby("sha256")["split"]
    .nunique()
    .gt(1)
    .sum()
)

print("Saved:", CORRECTED_PATH)


In [ ]:

# ============================================
# COMPARE OLD AND CORRECTED SPLITS
# ============================================

# Keep the original split_df before correction
# in a variable called original_split_df.
#
# If you haven't saved it yet, and split_df
# has already been overwritten, reload the
# original CSV created during Experiment 15.

original_split_df = pd.read_csv(
    "reports/eda/exp15_dataset_splits/"
    "dataset_split_manifest.csv"
)

corrected_split_df = pd.read_csv(
    "reports/eda/exp16_duplicate_detection/"
    "dataset_split_manifest_corrected.csv"
)

# Compare assignments using original S3 key
comparison = original_split_df[
    ["s3_key", "split"]
].merge(
    corrected_split_df[
        ["s3_key", "split"]
    ],
    on="s3_key",
    suffixes=("_before", "_after"),
    validate="one_to_one"
)

changed = comparison[
    comparison["split_before"] !=
    comparison["split_after"]
]

print("TOTAL IMAGES:", len(comparison))
print("Assignments changed:", len(changed))
print(
    "Percentage changed:",
    round(len(changed) / len(comparison) * 100, 2),
    "%"
)

print("\nBEFORE:")
display(
    original_split_df["split"]
    .value_counts()
    .to_frame("images")
)

print("\nAFTER:")
display(
    corrected_split_df["split"]
    .value_counts()
    .to_frame("images")
)

print("\nCHANGED ASSIGNMENTS:")
display(changed.head(20))

# Verify duplicate leakage is eliminated

check = corrected_split_df.merge(
    hash_df[["s3_key", "sha256"]],
    on="s3_key",
    validate="one_to_one"
)

remaining_leakage = (
    check.groupby("sha256")["split"]
    .nunique()
    .gt(1)
    .sum()
)

print(
    "\nRemaining cross-split duplicate groups:",
    remaining_leakage
)

assert remaining_leakage == 0
assert len(corrected_split_df) == 14565


In [ ]:

# ============================================
# FINAL DATASET SPLIT DISTRIBUTION
# ============================================

import pandas as pd
from pathlib import Path

CORRECTED_PATH = Path(
    "reports/eda/exp16_duplicate_detection/"
    "dataset_split_manifest_corrected.csv"
)

corrected_split_df = pd.read_csv(
    CORRECTED_PATH
)

assert len(corrected_split_df) == 14565
assert corrected_split_df["s3_key"].is_unique

split_order = [
    "train",
    "validation",
    "test"
]

print("1. OVERALL DISTRIBUTION")

display(
    corrected_split_df["split"]
    .value_counts()
    .reindex(split_order)
    .to_frame("image_count")
)

print("\n2. DEFECT-WISE DISTRIBUTION")

class_table = pd.crosstab(
    corrected_split_df["folder_2"],
    corrected_split_df["split"]
).reindex(columns=split_order)

display(class_table)

print("\n3. FABRIC-WISE DISTRIBUTION")

fabric_table = pd.crosstab(
    corrected_split_df["folder_1"],
    corrected_split_df["split"]
).reindex(columns=split_order)

display(fabric_table)

print("\n4. FABRIC × DEFECT DISTRIBUTION")

combination_table = pd.crosstab(
    [
        corrected_split_df["folder_1"],
        corrected_split_df["folder_2"]
    ],
    corrected_split_df["split"]
).reindex(columns=split_order)

display(combination_table)

print("\nNumber of combinations:", len(combination_table))

print(
    "Minimum images in a combination:",
    combination_table.sum(axis=1).min()
)

print("\nAll 121 combinations represented:",
      len(combination_table) == 121)

assert len(combination_table) == 121
assert (combination_table.sum(axis=1) > 0).all()

# Update active manifest
split_df = corrected_split_df.copy()

print("\nFinal corrected manifest loaded.")


In [ ]:

# ============================================
# EXPERIMENT 17
# COMPLETE PREPROCESSING - CONFIGURATION
# ============================================

import boto3
import cv2
import numpy as np
import pandas as pd

from pathlib import Path

# ============================================
# AWS CONFIGURATION
# ============================================

BUCKET_NAME = (
    "amal-stitchingnet-data-903473565738-ap-southeast-2-an"
)

REGION = "ap-southeast-2"

s3 = boto3.client(
    "s3",
    region_name=REGION
)

# ============================================
# LOAD CORRECTED MANIFEST
# ============================================

MANIFEST_PATH = Path(
    "reports/eda/exp16_duplicate_detection/"
    "dataset_split_manifest_corrected.csv"
)

processing_df = pd.read_csv(
    MANIFEST_PATH
)

assert len(processing_df) == 14565

assert processing_df["s3_key"].is_unique

# ============================================
# S3 OUTPUT STRUCTURE
# ============================================

SOURCE_PREFIX = "data/"

OUTPUT_ROOT = (
    "processed-data/exp17_multiview"
)

LAB_PREFIX = (
    f"{OUTPUT_ROOT}/lab_clahe"
)

BILATERAL_PREFIX = (
    f"{OUTPUT_ROOT}/bilateral_clahe"
)

# ============================================
# PROCESSING PARAMETERS
# ============================================

MAX_WORKERS = 2
BATCH_SIZE = 100

JPEG_QUALITY = 95

CLAHE_CLIP_LIMIT = 2.0
CLAHE_TILE_GRID = (8, 8)

BILATERAL_DIAMETER = 5
BILATERAL_SIGMA_COLOR = 30
BILATERAL_SIGMA_SPACE = 30

# Avoid CPU oversubscription
cv2.setNumThreads(1)

# ============================================
# OUTPUT PROGRESS FILES
# ============================================

PROGRESS_DIR = Path(
    "reports/eda/exp17_preprocessing"
)

PROGRESS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PROGRESS_FILE = (
    PROGRESS_DIR / "preprocessing_progress.csv"
)

ERROR_FILE = (
    PROGRESS_DIR / "preprocessing_errors.csv"
)

MANIFEST_FILE = (
    PROGRESS_DIR / "preprocessed_manifest.csv"
)

# ============================================
# CREATE ENHANCEMENT FUNCTIONS
# ============================================

clahe = cv2.createCLAHE(
    clipLimit=CLAHE_CLIP_LIMIT,
    tileGridSize=CLAHE_TILE_GRID
)


def apply_lab_clahe(bgr_image):

    lab = cv2.cvtColor(
        bgr_image,
        cv2.COLOR_BGR2LAB
    )

    l_channel, a_channel, b_channel = (
        cv2.split(lab)
    )

    enhanced_l = clahe.apply(
        l_channel
    )

    enhanced_lab = cv2.merge([
        enhanced_l,
        a_channel,
        b_channel
    ])

    return cv2.cvtColor(
        enhanced_lab,
        cv2.COLOR_LAB2BGR
    )


def apply_bilateral_clahe(bgr_image):

    lab = cv2.cvtColor(
        bgr_image,
        cv2.COLOR_BGR2LAB
    )

    l_channel, a_channel, b_channel = (
        cv2.split(lab)
    )

    filtered_l = cv2.bilateralFilter(
        l_channel,
        d=BILATERAL_DIAMETER,
        sigmaColor=BILATERAL_SIGMA_COLOR,
        sigmaSpace=BILATERAL_SIGMA_SPACE
    )

    enhanced_l = clahe.apply(
        filtered_l
    )

    enhanced_lab = cv2.merge([
        enhanced_l,
        a_channel,
        b_channel
    ])

    return cv2.cvtColor(
        enhanced_lab,
        cv2.COLOR_LAB2BGR
    )


# ============================================
# CONFIGURATION VERIFICATION
# ============================================

print("PREPROCESSING READY")

print("Source images:", len(processing_df))
print("Workers:", MAX_WORKERS)
print("Batch size:", BATCH_SIZE)

print("CLAHE:", CLAHE_CLIP_LIMIT,
      CLAHE_TILE_GRID)

print("Bilateral:",
      BILATERAL_DIAMETER,
      BILATERAL_SIGMA_COLOR,
      BILATERAL_SIGMA_SPACE)

print("Output:", OUTPUT_ROOT)

print("Original dataset will remain untouched.")


In [ ]:

# ============================================
# EXPERIMENT 17
# FULL DATASET PREPROCESSING
# ============================================

from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed
)

import mimetypes


# ============================================
# 1. LOAD PREVIOUS PROGRESS
# ============================================

if PROGRESS_FILE.exists():

    progress_df = pd.read_csv(
        PROGRESS_FILE
    )

    completed_keys = set(
        progress_df["s3_key"].astype(str)
    )

else:

    progress_df = pd.DataFrame()

    completed_keys = set()


pending_df = processing_df[
    ~processing_df["s3_key"].isin(
        completed_keys
    )
].copy()

print("PREPROCESSING STATUS")

print("Total images:", len(processing_df))
print("Already completed:", len(completed_keys))
print("Remaining:", len(pending_df))


# ============================================
# 2. ENCODE IMAGE
# ============================================

def encode_jpeg(image):

    success, encoded = cv2.imencode(
        ".jpg",
        image,
        [
            cv2.IMWRITE_JPEG_QUALITY,
            JPEG_QUALITY
        ]
    )

    if not success:

        raise ValueError(
            "JPEG encoding failed."
        )

    return encoded.tobytes()


# ============================================
# 3. PROCESS ONE SOURCE IMAGE
# ============================================

def process_one_image(row):

    source_key = row["s3_key"]

    # Read original S3 object

    response = s3.get_object(
        Bucket=BUCKET_NAME,
        Key=source_key
    )

    try:

        raw_bytes = response["Body"].read()

    finally:

        response["Body"].close()

    # Decode original image

    image_array = np.frombuffer(
        raw_bytes,
        dtype=np.uint8
    )

    original = cv2.imdecode(
        image_array,
        cv2.IMREAD_COLOR
    )

    if original is None:

        raise ValueError(
            f"Image decoding failed: {source_key}"
        )

    # Generate enhanced views

    lab_image = apply_lab_clahe(
        original
    )

    bilateral_image = apply_bilateral_clahe(
        original
    )

    # Encode enhanced images

    lab_bytes = encode_jpeg(
        lab_image
    )

    bilateral_bytes = encode_jpeg(
        bilateral_image
    )

    # Preserve original folder structure

    if not source_key.startswith(SOURCE_PREFIX):

        raise ValueError(
            f"Unexpected source key: {source_key}"
        )

    relative_key = source_key[
        len(SOURCE_PREFIX):
    ]

    lab_key = (
        f"{LAB_PREFIX}/{relative_key}"
    )

    bilateral_key = (
        f"{BILATERAL_PREFIX}/{relative_key}"
    )

    # Upload LAB CLAHE

    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=lab_key,
        Body=lab_bytes,
        ContentType="image/jpeg"
    )

    # Upload Bilateral + CLAHE

    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=bilateral_key,
        Body=bilateral_bytes,
        ContentType="image/jpeg"
    )

    # Return processing record

    return {

        "sample_id": row["sample_id"],

        "file_name": row["file_name"],

        "s3_key": source_key,

        "lab_clahe_s3_key": lab_key,

        "bilateral_clahe_s3_key": bilateral_key,

        "folder_1": row["folder_1"],

        "folder_2": row["folder_2"],

        "split": row["split"],

        "original_size_bytes": len(raw_bytes),

        "lab_size_bytes": len(lab_bytes),

        "bilateral_size_bytes": len(
            bilateral_bytes
        ),

        "height": original.shape[0],

        "width": original.shape[1],

        "status": "success"
    }


# ============================================
# 4. BATCH PROCESSING
# ============================================

all_errors = []

total_pending = len(pending_df)

for start in range(
    0,
    total_pending,
    BATCH_SIZE
):

    batch_df = pending_df.iloc[
        start:start + BATCH_SIZE
    ]

    batch_results = []

    batch_errors = []

    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        futures = {

            executor.submit(
                process_one_image,
                row
            ): row

            for row in batch_df.to_dict(
                "records"
            )
        }

        for future in as_completed(futures):

            row = futures[future]

            try:

                result = future.result()

                batch_results.append(
                    result
                )

            except Exception as error:

                batch_errors.append({

                    "s3_key": row["s3_key"],

                    "error": str(error)

                })

    # ========================================
    # 5. SAVE SUCCESSFUL PROGRESS
    # ========================================

    if batch_results:

        new_df = pd.DataFrame(
            batch_results
        )

        if progress_df.empty:

            progress_df = new_df

        else:

            progress_df = pd.concat(
                [
                    progress_df,
                    new_df
                ],
                ignore_index=True
            )

        progress_df = (
            progress_df
            .drop_duplicates(
                subset=["s3_key"],
                keep="last"
            )
        )

        progress_df.to_csv(
            PROGRESS_FILE,
            index=False
        )

    # ========================================
    # 6. SAVE ERRORS
    # ========================================

    all_errors.extend(
        batch_errors
    )

    if all_errors:

        pd.DataFrame(
            all_errors
        ).to_csv(
            ERROR_FILE,
            index=False
        )

    # ========================================
    # 7. PROGRESS REPORT
    # ========================================

    processed_count = min(
        start + len(batch_df),
        total_pending
    )

    print(
        f"Batch completed: "
        f"{processed_count}/{total_pending} "
        f"| Total successful: {len(progress_df)} "
        f"| Errors: {len(all_errors)}"
    )


# ============================================
# 8. SAVE FINAL MANIFEST
# ============================================

if not progress_df.empty:

    progress_df = (
        progress_df
        .drop_duplicates(
            subset=["s3_key"],
            keep="last"
        )
    )

    progress_df.to_csv(
        MANIFEST_FILE,
        index=False
    )


# ============================================
# 9. FINAL SUMMARY
# ============================================

successful = len(progress_df)

remaining = (
    len(processing_df)
    - successful
)

print("\nFULL PREPROCESSING SUMMARY")

print("Source images:", len(processing_df))

print("Successfully processed:", successful)

print("Remaining:", remaining)

print("Errors:", len(all_errors))

print("Manifest:", MANIFEST_FILE)

if remaining == 0 and len(all_errors) == 0:

    print("\nALL IMAGES PROCESSED SUCCESSFULLY")

else:

    print("\nPROCESSING NOT YET COMPLETE")

    print(
        "Resolve errors and rerun this cell."
    )


In [ ]:

# ============================================
# EXPERIMENT 17
# FINAL S3 PREPROCESSING VERIFICATION
# ============================================

assert "progress_df" in globals()

assert len(progress_df) == 14565, (
    "Not all source images were processed."
)

# ============================================
# 1. LIST S3 OBJECTS
# ============================================

def list_s3_keys(prefix):

    keys = []

    paginator = s3.get_paginator(
        "list_objects_v2"
    )

    for page in paginator.paginate(
        Bucket=BUCKET_NAME,
        Prefix=prefix
    ):

        for obj in page.get("Contents", []):

            if not obj["Key"].endswith("/"):

                keys.append(
                    obj["Key"]
                )

    return set(keys)


print("Listing LAB CLAHE objects...")

actual_lab = list_s3_keys(
    LAB_PREFIX + "/"
)

print("Listing Bilateral CLAHE objects...")

actual_bilateral = list_s3_keys(
    BILATERAL_PREFIX + "/"
)

# ============================================
# 2. EXPECTED OUTPUT KEYS
# ============================================

expected_lab = set(
    progress_df["lab_clahe_s3_key"]
)

expected_bilateral = set(
    progress_df["bilateral_clahe_s3_key"]
)

missing_lab = (
    expected_lab - actual_lab
)

missing_bilateral = (
    expected_bilateral - actual_bilateral
)

# ============================================
# 3. OUTPUT COUNT CHECK
# ============================================

print("\nS3 OUTPUT VERIFICATION")
print("======================")

print("Expected LAB:", len(expected_lab))
print("Actual LAB:", len(actual_lab))

print(
    "Missing LAB:",
    len(missing_lab)
)

print(
    "Expected Bilateral:",
    len(expected_bilateral)
)

print(
    "Actual Bilateral:",
    len(actual_bilateral)
)

print(
    "Missing Bilateral:",
    len(missing_bilateral)
)

# ============================================
# 4. VERIFY STORED IMAGE DECODING
# ============================================

sample_df = progress_df.sample(
    n=min(20, len(progress_df)),
    random_state=42
)

decode_errors = []

for _, row in sample_df.iterrows():

    for key_column in [
        "lab_clahe_s3_key",
        "bilateral_clahe_s3_key"
    ]:

        key = row[key_column]

        response = s3.get_object(
            Bucket=BUCKET_NAME,
            Key=key
        )

        try:

            raw = response["Body"].read()

        finally:

            response["Body"].close()

        decoded = cv2.imdecode(
            np.frombuffer(
                raw,
                dtype=np.uint8
            ),
            cv2.IMREAD_COLOR
        )

        if decoded is None:

            decode_errors.append(key)

        elif decoded.shape[:2] != (
            int(row["height"]),
            int(row["width"])
        ):

            decode_errors.append(key)


# ============================================
# 5. FINAL VALIDATION
# ============================================

print("\nIMAGE DECODING ERRORS:",
      len(decode_errors))

print(
    "LAB storage (GB):",
    round(
        sum(
            progress_df["lab_size_bytes"]
        ) / (1024 ** 3),
        3
    )
)

print(
    "Bilateral storage (GB):",
    round(
        sum(
            progress_df["bilateral_size_bytes"]
        ) / (1024 ** 3),
        3
    )
)

assert len(missing_lab) == 0
assert len(missing_bilateral) == 0
assert len(decode_errors) == 0

print("\nALL PREPROCESSING VERIFICATIONS PASSED")
